In [4]:
!python -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------------- ---------- 1.3/1.8 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 10.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [5]:
pip install requests beautifulsoup4 lxml pandas


Note: you may need to restart the kernel to use updated packages.


In [7]:
import re
import time
from datetime import date, datetime
from urllib.parse import urlencode, urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://kncpain.com/bbs/board.php"
BO_TABLE = "free"

SITE_NAME = "kncpain"
START_DATE = date(2026, 1, 1)
END_DATE = date.today()   # "지금/최근까지" → 오늘 기준

SLEEP_SEC = 0.25
MAX_PAGES = 500

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

def get_soup(session: requests.Session, url: str) -> BeautifulSoup:
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def clean_text(x: str | None) -> str | None:
    if not x:
        return None
    x = re.sub(r"\s+", " ", x.strip())
    return x if x else None

def parse_full_date(raw: str) -> date | None:
    """YYYY-MM-DD / YYYY.MM.DD 등"""
    if not raw:
        return None
    m = re.search(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})", raw)
    if not m:
        return None
    y, mo, d = map(int, m.groups())
    return date(y, mo, d)

def parse_mmdd(raw: str) -> tuple[int,int] | None:
    """MM-DD / MM.DD 등 (연도 없음)"""
    if not raw:
        return None
    m = re.fullmatch(r"\s*(\d{1,2})[.\-\/](\d{1,2})\s*", raw)
    if not m:
        return None
    mo, d = map(int, m.groups())
    return (mo, d)

def is_time_only(raw: str) -> bool:
    return bool(raw and re.fullmatch(r"\s*\d{1,2}:\d{2}\s*", raw))

def find_table_and_col_index(soup: BeautifulSoup):
    """
    게시판 스킨이 달라도,
    th 텍스트(제목/날짜/조회)를 보고 해당 td 인덱스를 찾아냅니다.
    """
    table = soup.select_one("#bo_list table") or soup.select_one("#bo_list") or soup.select_one("table")
    if not table:
        raise RuntimeError("게시판 테이블을 찾지 못했습니다. (#bo_list/table 구조 확인 필요)")

    # 헤더(th) 찾기
    thead = table.select_one("thead")
    ths = thead.select("th") if thead else []
    header_texts = [clean_text(th.get_text()) or "" for th in ths]

    def find_idx(keywords: list[str]) -> int | None:
        for i, ht in enumerate(header_texts):
            for kw in keywords:
                if kw in ht:
                    return i
        return None

    # 그누보드에서 보통 "제목", "날짜", "조회" 같은 텍스트가 있음
    idx_subject = find_idx(["제목", "subject"])  # 제목
    idx_date = find_idx(["날짜", "작성일", "date"])  # 날짜
    idx_view = find_idx(["조회", "조회수", "view"])  # 조회

    # 만약 헤더에서 못 찾으면(스킨 특이) → 안전장치(대충 추정)
    # 제목은 보통 가운데, 날짜/조회는 뒤쪽
    ncols = len(header_texts)
    if idx_subject is None and ncols > 0:
        idx_subject = min(1, ncols - 1)  # 보통 2번째 칸이 제목
    if idx_date is None and ncols > 0:
        idx_date = ncols - 2
    if idx_view is None and ncols > 0:
        idx_view = ncols - 1

    tbody = table.select_one("tbody")
    if not tbody:
        raise RuntimeError("tbody를 찾지 못했습니다. 테이블 구조 확인 필요")

    return table, tbody, idx_subject, idx_date, idx_view

def crawl_kncpain_between(start_date: date, end_date: date) -> pd.DataFrame:
    session = requests.Session()
    results = []

    # 연도 없는 "MM-DD"를 정확히 복원하기 위한 상태
    # 최신글부터 과거로 내려갈수록, (MM-DD)가 갑자기 커지면 연도가 한 해 내려간 것으로 판단
    inferred_year = end_date.year
    last_mmdd = None  # (mo, d)

    stopped = False

    for page in range(1, MAX_PAGES + 1):
        url = f"{BASE}?{urlencode({'bo_table': BO_TABLE, 'page': page})}"
        soup = get_soup(session, url)

        table, tbody, idx_subject, idx_date, idx_view = find_table_and_col_index(soup)
        rows = tbody.select("tr")
        if not rows:
            break

        for tr in rows:
            tds = tr.select("td")
            if not tds:
                continue

            # 제목/링크
            subj_td = tds[idx_subject] if idx_subject is not None and idx_subject < len(tds) else None
            a = subj_td.select_one("a") if subj_td else tr.select_one("a")
            if not a:
                continue

            title = clean_text(a.get_text(" ", strip=True))
            href = a.get("href")
            full_url = urljoin(url, href) if href else url

            # raw_date
            date_td = tds[idx_date] if idx_date is not None and idx_date < len(tds) else None
            raw_date = clean_text(date_td.get_text(" ", strip=True) if date_td else None)

            # view
            view_td = tds[idx_view] if idx_view is not None and idx_view < len(tds) else None
            view_raw = clean_text(view_td.get_text(" ", strip=True) if view_td else None)
            view = None
            if view_raw:
                digits = "".join(ch for ch in view_raw if ch.isdigit())
                view = int(digits) if digits else None

            # 날짜 정규화(date)
            normalized = None
            d_full = parse_full_date(raw_date or "")
            if d_full:
                normalized = d_full
                # full date가 나오면 inferred_year도 동기화(안전)
                inferred_year = d_full.year
                last_mmdd = (d_full.month, d_full.day)
            else:
                if is_time_only(raw_date or ""):
                    # '16:24' 같은 형태면 "오늘"로 간주
                    normalized = end_date
                else:
                    mmdd = parse_mmdd(raw_date or "")
                    if mmdd:
                        # 연도 추론: 최신→과거로 내려가는데 MM-DD가 갑자기 커지면 연도 -1
                        if last_mmdd is not None and mmdd > last_mmdd:
                            inferred_year -= 1
                        normalized = date(inferred_year, mmdd[0], mmdd[1])
                        last_mmdd = mmdd

            if not normalized:
                # 날짜를 못 읽으면 일단 스킵(원하면 여기서 상세페이지 파싱 추가 가능)
                continue

            # 범위 필터
            if normalized < start_date:
                stopped = True
                break
            if normalized > end_date:
                # 미래/이상치면 스킵
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": raw_date,
                "date": normalized.strftime("%Y-%m-%d"),
                "view": view,
                "url": full_url,
            })

        if stopped:
            break

        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    # 최신순 정렬(원하면)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values(["date"], ascending=False)
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


if __name__ == "__main__":
    df = crawl_kncpain_between(START_DATE, END_DATE)
    out = "kncpain_free_2026_recent.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print("saved:", out, "rows:", len(df))
    if len(df) > 0:
        print(df.head(5).to_string(index=False))


saved: kncpain_free_2026_recent.csv rows: 7500
   site                                                         title raw_date       date  view                                                                  url
kncpain               ◆토지노수류탄 BOMB-7.COM 가입코드 A77◆ 지인추천221만원 입금플러스50%    20:54 2026-02-14     1  https://kncpain.com/bbs/board.php?bo_table=free&wr_id=171561&page=1
kncpain                                  하나약국 정품 | 성기능증진약【ssww99.xyz】    09:30 2026-02-14     1 https://kncpain.com/bbs/board.php?bo_table=free&wr_id=171376&page=13
kncpain 토지노수류탄 ★ 무한매충20% 슬롯매충10% 토지노사이트추천 스마일주소 스마일도메인 스마일검증 ★ 【토지노수…    07:51 2026-02-14     1 https://kncpain.com/bbs/board.php?bo_table=free&wr_id=171360&page=14
kncpain 수류탄 bomb-toto.com 가입코드:A77 신규 첫충 50% 3+2 10+5 20+8 30+12 50+…    07:51 2026-02-14     1 https://kncpain.com/bbs/board.php?bo_table=free&wr_id=171361&page=14
kncpain       ■수류탄 BOMB-7.COM 코드 A77■ 신규가입 첫충 40% 매충 10% 돌발 15% 무제한지급    07:51 2026-02-14     1 https://kncpain.com/bbs/board.ph

In [9]:
import re
import time
from datetime import date
from urllib.parse import urlencode, urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"
BO_TABLE = "free"

SITE_NAME = "kncpain"
START_DATE = date(2026, 1, 1)
END_DATE = date.today()  # 오늘까지(최근까지)

SLEEP_SEC = 0.25

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

def get_soup(session: requests.Session, url: str) -> BeautifulSoup:
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def clean_text(x: str | None) -> str | None:
    if not x:
        return None
    x = re.sub(r"\s+", " ", x.strip())
    return x if x else None

def parse_full_date(raw: str) -> date | None:
    if not raw:
        return None
    m = re.search(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})", raw)
    if not m:
        return None
    y, mo, d = map(int, m.groups())
    return date(y, mo, d)

def parse_mmdd(raw: str) -> tuple[int, int] | None:
    if not raw:
        return None
    m = re.fullmatch(r"\s*(\d{1,2})[.\-\/](\d{1,2})\s*", raw)
    if not m:
        return None
    mo, d = map(int, m.groups())
    return (mo, d)

def is_time_only(raw: str) -> bool:
    return bool(raw and re.fullmatch(r"\s*\d{1,2}:\d{2}\s*", raw))

def find_table_and_col_index(soup: BeautifulSoup):
    table = soup.select_one("#bo_list table") or soup.select_one("#bo_list") or soup.select_one("table")
    if not table:
        raise RuntimeError("게시판 테이블을 찾지 못했습니다. (#bo_list/table 구조 확인 필요)")

    thead = table.select_one("thead")
    ths = thead.select("th") if thead else []
    header_texts = [clean_text(th.get_text()) or "" for th in ths]

    def find_idx(keywords: list[str]) -> int | None:
        for i, ht in enumerate(header_texts):
            for kw in keywords:
                if kw in ht:
                    return i
        return None

    idx_subject = find_idx(["제목", "subject"])
    idx_date = find_idx(["날짜", "작성일", "date"])
    idx_view = find_idx(["조회", "조회수", "view"])

    ncols = len(header_texts)
    if idx_subject is None and ncols > 0:
        idx_subject = min(1, ncols - 1)
    if idx_date is None and ncols > 0:
        idx_date = ncols - 2
    if idx_view is None and ncols > 0:
        idx_view = ncols - 1

    tbody = table.select_one("tbody")
    if not tbody:
        raise RuntimeError("tbody를 찾지 못했습니다. 테이블 구조 확인 필요")

    return tbody, idx_subject, idx_date, idx_view

def crawl_from_20260101() -> pd.DataFrame:
    session = requests.Session()
    results = []

    inferred_year = END_DATE.year
    last_mmdd = None

    page = 1
    while True:
        url = f"{BASE}?{urlencode({'bo_table': BO_TABLE, 'page': page})}"
        soup = get_soup(session, url)

        tbody, idx_subject, idx_date, idx_view = find_table_and_col_index(soup)
        rows = tbody.select("tr")

        # ✅ 더 이상 페이지가 없으면 종료
        if not rows:
            break

        stop_all = False

        for tr in rows:
            tds = tr.select("td")
            if not tds:
                continue

            # 제목/링크
            subj_td = tds[idx_subject] if idx_subject is not None and idx_subject < len(tds) else None
            a = subj_td.select_one("a") if subj_td else tr.select_one("a")
            if not a:
                continue

            title = clean_text(a.get_text(" ", strip=True))
            href = a.get("href")
            full_url = urljoin(url, href) if href else url

            # raw_date
            date_td = tds[idx_date] if idx_date is not None and idx_date < len(tds) else None
            raw_date = clean_text(date_td.get_text(" ", strip=True) if date_td else None)

            # view
            view_td = tds[idx_view] if idx_view is not None and idx_view < len(tds) else None
            view_raw = clean_text(view_td.get_text(" ", strip=True) if view_td else None)
            view = None
            if view_raw:
                digits = "".join(ch for ch in view_raw if ch.isdigit())
                view = int(digits) if digits else None

            # 날짜 정규화
            normalized = None
            d_full = parse_full_date(raw_date or "")
            if d_full:
                normalized = d_full
                inferred_year = d_full.year
                last_mmdd = (d_full.month, d_full.day)
            else:
                if is_time_only(raw_date or ""):
                    normalized = END_DATE
                else:
                    mmdd = parse_mmdd(raw_date or "")
                    if mmdd:
                        if last_mmdd is not None and mmdd > last_mmdd:
                            inferred_year -= 1
                        normalized = date(inferred_year, mmdd[0], mmdd[1])
                        last_mmdd = mmdd

            if not normalized:
                continue

            # ✅ 범위 밖이면 처리
            if normalized < START_DATE:
                stop_all = True
                break
            if normalized > END_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": raw_date,
                "date": normalized.strftime("%Y-%m-%d"),
                "view": view,
                "url": full_url,
            })

        # ✅ 2026-01-01 이전 글이 나오면 전체 종료
        if stop_all:
            break

        page += 1
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values(["date"], ascending=False)
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_from_20260101()
    out = "kncpain_free_from_20260101.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print("saved:", out, "rows:", len(df))
    if len(df) > 0:
        print(df.head(5).to_string(index=False))


RuntimeError: 게시판 테이블을 찾지 못했습니다. (#bo_list/table 구조 확인 필요)

In [11]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

LIST_URL = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"
SITE_NAME = "시트라인"  # 원하시면 "xn--oi2..." 같은 영문으로 바꿔도 됩니다.

START_DATE = date(2026, 1, 1)
END_DATE = date.today()

SLEEP_SEC = 0.25

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

# ✅ 이 사이트 글 상세 URL 패턴(실제로 이렇게 열림) :contentReference[oaicite:1]{index=1}
ARTICLE_HREF_RE = re.compile(r"^/article/[^/]+/6/\d+/?$")

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")

def get_soup(session: requests.Session, url: str) -> BeautifulSoup:
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def parse_detail_date_and_view(detail_soup: BeautifulSoup):
    """
    상세페이지 텍스트에 '작성일 2026-02-14', '조회 0' 같은 형태가 존재함 :contentReference[oaicite:2]{index=2}
    → 정규식으로 날짜/조회 뽑기
    """
    text = detail_soup.get_text(" ", strip=True)

    # 작성일: 페이지 어딘가에 날짜가 여러 번 나올 수 있으니,
    # '작성일' 근처를 우선 노리는 방식도 가능하지만 여기선 간단히 "첫 YYYY-MM-DD"를 사용
    m = DATE_RE.search(text)
    if not m:
        return None, None, None

    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = None
    if m2:
        view = int(m2.group(1).replace(",", ""))

    # raw_date는 상세페이지에서는 보통 날짜 전체가 있으니 그대로 date_str로 채움
    return d_obj, d_str, view

def make_page_url(base: str, page: int) -> str:
    # 카페24 게시판은 보통 ?page=2 같은 방식이 흔합니다.
    # 만약 이게 안 먹으면, 아래 2줄 중 하나로 바꿔서 쓰시면 됩니다.
    # return f"{base}?board_no=6&page={page}"
    # return f"{base}?page={page}"
    return f"{base}?page={page}"

def crawl_from_20260101_unlimited() -> pd.DataFrame:
    session = requests.Session()
    results = []
    seen = set()

    page = 1
    while True:
        list_url = make_page_url(LIST_URL, page)
        soup = get_soup(session, list_url)

        # 목록에서 글 링크 추출 (상세 URL 패턴에 맞는 것만)
        anchors = soup.select("a[href]")
        article_links = []
        for a in anchors:
            href = a.get("href", "").strip()
            if ARTICLE_HREF_RE.match(href):
                full = urljoin(LIST_URL, href)
                title = a.get_text(" ", strip=True)
                if full and full not in seen and title:
                    article_links.append((title, full))

        # ✅ 글 링크가 없으면 더 이상 페이지가 없거나 구조가 다름 → 종료
        if not article_links:
            break

        stop_all = False

        for title, url in article_links:
            if url in seen:
                continue
            seen.add(url)

            detail_soup = get_soup(session, url)
            d_obj, d_str, view = parse_detail_date_and_view(detail_soup)

            # 날짜 못 읽으면 스킵
            if not d_obj:
                continue

            # 범위 필터 (최신→과거로 내려간다는 전제)
            if d_obj < START_DATE:
                stop_all = True
                break
            if d_obj > END_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,   # 이 사이트는 상세에서 YYYY-MM-DD 확보 가능 :contentReference[oaicite:3]{index=3}
                "date": d_str,
                "view": view,
                "url": url,
            })

            time.sleep(SLEEP_SEC)

        if stop_all:
            break

        page += 1
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=False)
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_from_20260101_unlimited()
    out = "sheetline_qa_2026_recent.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print("saved:", out, "rows:", len(df))
    if len(df) > 0:
        print(df.head(5).to_string(index=False))


saved: sheetline_qa_2026_recent.csv rows: 0


In [12]:
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

LIST_URL = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    print("GET", url, "status", r.status_code, "len", len(r.text))
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

# ✅ 정규식 느슨하게: /article/ + /6/ + 글번호만 있으면 OK
def is_article_href(href: str) -> bool:
    if not href:
        return False
    return ("/article/" in href) and ("/6/" in href) and re.search(r"/6/\d+", href)

soup = get_soup(LIST_URL)

# 목록에서 글 링크를 “선택자 기반”으로 먼저 시도 (카페24에서 흔한 패턴)
anchors = soup.select("tr.xans-record- .subject a[href], .subject a[href], a[href]")
links = []
for a in anchors:
    href = a.get("href", "").strip()
    if is_article_href(href):
        full = urljoin(LIST_URL, href)
        title = a.get_text(" ", strip=True)
        if title:
            links.append((title, full))

# 중복 제거
uniq = []
seen = set()
for t,u in links:
    if u not in seen:
        seen.add(u)
        uniq.append((t,u))

print("✅ 1페이지에서 잡힌 글 링크 수:", len(uniq))
print("샘플 5개:")
for t,u in uniq[:5]:
    print("-", t, u)

print("\n✅ 상세페이지 날짜 샘플(첫 5개):")
for t,u in uniq[:5]:
    ds = get_soup(u).get_text(" ", strip=True)
    m = DATE_RE.search(ds)
    print("-", t, "=>", (m.group(0) if m else "날짜 못찾음"))


GET https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/ status 200 len 142560
✅ 1페이지에서 잡힌 글 링크 수: 15
샘플 5개:
- 50678 네이버아이디판매'구매'/텔:@cupid4989/'틱톡아이디판매'페이스북판매'외국인'리뷰'계정'인스타그램아이디'판매'트워터아이디'휴면아이디판매'/TG:@xinxiandb/' T**** 0 0 2026-02-14 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86520/
- 50677 월곡동다국적노래클럽 ※010-7903-4858※ 비아동노래홀이벤트 운남동쓰리노 광주룸싸롱2차금액 이**** 0 0 2026-02-14 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86519/
- 50676 양산동다국적노래클럽 ↪010+7903+4858↩ 광주백운동풀싸롱왕복픽업 효덕동풀싸롱 첨단지구러시아노래클럽연말 이**** 0 0 2026-02-14 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86518/
- 50675 어룡동풀싸롱 ▨O1O=7903=4858▨ 충장동노래홀혼자 광주백운동풀싸롱 문화동러시아노래클럽이벤트 이**** 0 0 2026-02-14 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86517/
- 50674 매곡동하드터치룸 ⌊ 010+7903+4858 ⌉ 수완지구하이퍼블릭예약방법 농성동2차노래방 용봉동러시아노래클럽골프모임 이**** 0 0 2026-02-14 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86516/

✅ 상세페이지 날짜 샘플(첫 5개):
GET https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86520/ status 200 len 132762
- 50678 네이버아이디판매'구매'/텔:@cupid4989/'틱톡아이디판

In [13]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_LIST = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"
SITE_NAME = "시트라인"   # 원하시면 다른 이름으로 변경

START_DATE = date(2026, 1, 1)
END_DATE = date.today()  # 오늘/최근까지

SLEEP_SEC = 0.3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")
ARTICLE_RE = re.compile(r"/article/.+/6/\d+")

def get_soup(session, url):
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def make_page_url(page: int) -> str:
    # ✅ 여기 중요: 이 사이트는 board_no + page 조합이 잘 먹습니다
    return f"{BASE_LIST}?board_no=6&page={page}"

def parse_detail(detail_soup):
    text = detail_soup.get_text(" ", strip=True)

    m = DATE_RE.search(text)
    if not m:
        return None, None

    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = int(m2.group(1).replace(",", "")) if m2 else None
    return d_obj, (d_str, view)

def crawl_from_20260101_unlimited():
    session = requests.Session()
    results = []
    seen = set()

    page = 1
    while True:
        list_url = make_page_url(page)
        soup = get_soup(session, list_url)

        # ✅ 목록 링크 추출(지금 디버그에서 15개 잘 잡힌 방식)
        anchors = soup.select("tr.xans-record- .subject a[href], .subject a[href]")
        items = []
        for a in anchors:
            href = (a.get("href") or "").strip()
            if not href:
                continue
            if not ARTICLE_RE.search(href):
                continue

            full = urljoin(BASE_LIST, href)
            title = a.get_text(" ", strip=True)
            if title and full not in seen:
                items.append((title, full))

        # ✅ 이 페이지에 글 링크가 없으면 마지막 페이지로 간주하고 종료
        if not items:
            break

        stop_all = False
        for title, url in items:
            if url in seen:
                continue
            seen.add(url)

            d_obj, pack = parse_detail(get_soup(session, url))
            if not d_obj or not pack:
                continue

            d_str, view = pack

            # ✅ 날짜 범위 필터
            if d_obj < START_DATE:
                stop_all = True
                break
            if d_obj > END_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,  # 이 사이트는 상세에서 YYYY-MM-DD 확정 가능
                "date": d_str,
                "view": view,
                "url": url,
            })

            time.sleep(SLEEP_SEC)

        if stop_all:
            break

        page += 1
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=False)
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


if __name__ == "__main__":
    df = crawl_from_20260101_unlimited()
    print("rows:", len(df))

    # ✅ CSV + XLSX 둘 다 저장
    df.to_csv("sheetline_qa_from_20260101.csv", index=False, encoding="utf-8-sig")
    df.to_excel("sheetline_qa_from_20260101.xlsx", index=False)

    print("saved: sheetline_qa_from_20260101.csv / sheetline_qa_from_20260101.xlsx")
    if len(df) > 0:
        print(df.head(5).to_string(index=False))


rows: 1077
saved: sheetline_qa_from_20260101.csv / sheetline_qa_from_20260101.xlsx
site                                                                                                                         title   raw_date       date  view                                                         url
시트라인                               네이버아이디판매'구매'/텔:@cupid4989/'틱톡아이디판매'페이스북판매'외국인'리뷰'계정'인스타그램아이디'판매'트워터아이디'휴면아이디판매'/TG:@xinxiandb/' 2026-02-14 2026-02-14     2 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86520/page/1/
시트라인                                                                         주월동풀싸롱 ⌊ 010-7903-4858 ⌋ 유덕동노래홀애프터 신용동쓰리노 상무지구하드터치룸터치 2026-02-14 2026-02-14     1 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86448/page/4/
시트라인 병원진단서위조【상담 ㉸아톡♥: hoy29 ▶텔레상담:+82 10-2312-6947】◈ #토익성적표제작 #허위공문서위조 #국가기술자격증위조 #위조 #수정 #제작 수정업체-제작업체-위조업체-대리시험  ▣ 고객님께 철통 보안과 안 2026-02-14 2026-02-14     1 https://xn--oi2bs9sguddvo.com/article/상품-qa/6/86462/page/3/
시트라인 #민증제작【상담 ㉸아톡♥: hoy29 ▶텔레상담:+82 10-23

In [16]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_LIST = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"
SITE_NAME = "시트라인"

START_DATE = date(2026, 1, 1)
END_DATE = date.today()

SLEEP_SEC = 0.3

# ✅ "링크 0개 페이지"가 나와도 바로 종료하지 않기
MAX_EMPTY_PAGES_IN_A_ROW = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")
ARTICLE_RE = re.compile(r"/article/.+/6/\d+")  # 느슨하게

def get_soup(session, url):
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def make_page_url(page: int) -> str:
    # ✅ 사용자가 확인한 규칙
    return f"{BASE_LIST}?board_no=6&page={page}"

def parse_detail(detail_soup):
    text = detail_soup.get_text(" ", strip=True)

    m = DATE_RE.search(text)
    if not m:
        return None, None

    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = int(m2.group(1).replace(",", "")) if m2 else None
    return d_obj, (d_str, view)

def extract_article_items(list_soup, base_url):
    """
    ✅ 구조가 바뀌어도 살아남는 '광역 수집' 방식:
    - 페이지 전체에서 /article/ ... /6/글번호 형태 링크를 다 긁고
    - 제목 텍스트가 있는 것만 남김
    """
    anchors = list_soup.select('a[href*="/article/"][href*="/6/"]')
    items = []
    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href:
            continue
        if not ARTICLE_RE.search(href):
            continue

        full = urljoin(base_url, href)
        title = a.get_text(" ", strip=True)
        if title:
            items.append((title, full))

    # ✅ 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for t, u in items:
        if u not in seen:
            seen.add(u)
            uniq.append((t, u))
    return uniq

def crawl_from_20260101_unlimited():
    session = requests.Session()
    results = []
    seen_articles = set()

    page = 1
    empty_streak = 0

    while True:
        list_url = make_page_url(page)
        soup = get_soup(session, list_url)

        items = extract_article_items(soup, BASE_LIST)

        # ✅ 링크가 0개면 바로 종료하지 말고 몇 페이지 더 시도
        if not items:
            empty_streak += 1
            print(f"[page {page}] items=0 (empty_streak={empty_streak}) url={list_url}")
            if empty_streak >= MAX_EMPTY_PAGES_IN_A_ROW:
                print("연속으로 링크가 없는 페이지가 발생해 종료합니다.")
                break
            page += 1
            time.sleep(SLEEP_SEC)
            continue
        else:
            empty_streak = 0

        stop_all = False
        kept_this_page = 0

        for title, url in items:
            if url in seen_articles:
                continue
            seen_articles.add(url)

            detail_soup = get_soup(session, url)
            d_obj, pack = parse_detail(detail_soup)
            if not d_obj or not pack:
                continue

            d_str, view = pack

            # ✅ 날짜 범위 필터
            if d_obj < START_DATE:
                stop_all = True
                break
            if d_obj > END_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,
                "date": d_str,
                "view": view,
                "url": url,
            })
            kept_this_page += 1
            time.sleep(SLEEP_SEC)

        print(f"[page {page}] items={len(items)} kept={kept_this_page} url={list_url}")

        if stop_all:
            print("START_DATE(2026-01-01)보다 과거 글을 만나 종료합니다.")
            break

        page += 1
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=False)
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_from_20260101_unlimited()
    print("rows:", len(df))

    df.to_csv("sheetline_qa_from_20260101.csv", index=False, encoding="utf-8-sig")
    df.to_excel("sheetline_qa_from_20260101.xlsx", index=False)  # openpyxl 설치 필요

    if len(df) > 0:
        print("min_date:", df["date"].min(), "max_date:", df["date"].max())
    print("saved: sheetline_qa_from_20260101.csv / sheetline_qa_from_20260101.xlsx")


[page 1] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=1
[page 2] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=2
[page 3] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=3
[page 4] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=4
[page 5] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=5
[page 6] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=6
[page 7] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=7
[page 8] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=8
[page 9] items=15 kept=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=9
[page 10] items=15 

In [17]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_LIST = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/"
SITE_NAME = "시트라인"

# ✅ 원하는 구간만
RANGE_START = date(2026, 1, 1)
RANGE_END   = date(2026, 1, 3)

# ✅ 여기까지만 페이지 크롤링 (사용자 확인: page=179가 1/3)
END_PAGE = 179

SLEEP_SEC = 0.25

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")
ARTICLE_RE = re.compile(r"/article/.+/6/\d+")

def get_soup(session, url):
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def make_page_url(page: int) -> str:
    return f"{BASE_LIST}?board_no=6&page={page}"

def parse_detail(detail_soup):
    text = detail_soup.get_text(" ", strip=True)
    m = DATE_RE.search(text)
    if not m:
        return None, None

    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = int(m2.group(1).replace(",", "")) if m2 else None
    return d_obj, (d_str, view)

def extract_article_items(list_soup, base_url):
    anchors = list_soup.select('a[href*="/article/"][href*="/6/"]')
    items = []
    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href:
            continue
        if not ARTICLE_RE.search(href):
            continue

        full = urljoin(base_url, href)
        title = a.get_text(" ", strip=True)
        if title:
            items.append((title, full))

    # 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for t, u in items:
        if u not in seen:
            seen.add(u)
            uniq.append((t, u))
    return uniq

def crawl_range_until_page(end_page: int):
    session = requests.Session()
    results = []
    seen_articles = set()

    for page in range(1, end_page + 1):
        list_url = make_page_url(page)
        soup = get_soup(session, list_url)
        items = extract_article_items(soup, BASE_LIST)

        if not items:
            print(f"[page {page}] items=0 url={list_url} (스킵)")
            continue

        kept = 0
        for title, url in items:
            if url in seen_articles:
                continue
            seen_articles.add(url)

            d_obj, pack = parse_detail(get_soup(session, url))
            if not d_obj or not pack:
                continue

            d_str, view = pack

            # ✅ 여기서 “딱 1/1~1/3만” 수집
            if d_obj < RANGE_START or d_obj > RANGE_END:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,
                "date": d_str,
                "view": view,
                "url": url,
            })
            kept += 1
            time.sleep(SLEEP_SEC)

        print(f"[page {page}] items={len(items)} kept_in_range={kept} url={list_url}")
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site", "title", "raw_date", "date", "view", "url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=True)  # 1/1→1/3 순으로 보고 싶으면 ascending=True
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_range_until_page(END_PAGE)
    print("rows:", len(df))
    if len(df) > 0:
        print("min_date:", df["date"].min(), "max_date:", df["date"].max())

    df.to_csv("sheetline_qa_20260101_20260103.csv", index=False, encoding="utf-8-sig")
    df.to_excel("sheetline_qa_20260101_20260103.xlsx", index=False)

    print("saved: sheetline_qa_20260101_20260103.csv / .xlsx")


[page 1] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=1 (스킵)
[page 2] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=2 (스킵)
[page 3] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=3 (스킵)
[page 4] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=4 (스킵)
[page 5] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=5 (스킵)
[page 6] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=6 (스킵)
[page 7] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=7 (스킵)
[page 8] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=8 (스킵)
[page 9] items=0 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=9 (스킵)
[page 10] items=0 url=https://xn--oi2bs9sguddvo.com/boa

In [18]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

START_URL = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179"
SITE_NAME = "시트라인"

TARGET_MIN_DATE = date(2026, 1, 1)   # ✅ 여기까지 내려가면 종료
TARGET_MAX_DATE = date(2026, 1, 3)   # ✅ 179페이지가 1/3이라 상한(원치 않으면 date.today()로 바꾸세요)

SLEEP_SEC = 0.35

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")
ARTICLE_RE = re.compile(r"/article/.+/6/\d+")

def get_soup(session, url):
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def parse_detail(detail_soup):
    text = detail_soup.get_text(" ", strip=True)
    m = DATE_RE.search(text)
    if not m:
        return None, None
    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = int(m2.group(1).replace(",", "")) if m2 else None
    return d_obj, (d_str, view)

def extract_article_items(list_soup, base_url):
    anchors = list_soup.select('a[href*="/article/"][href*="/6/"]')
    items = []
    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href or not ARTICLE_RE.search(href):
            continue
        full = urljoin(base_url, href)
        title = a.get_text(" ", strip=True)
        if title:
            items.append((title, full))

    # 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for t, u in items:
        if u not in seen:
            seen.add(u)
            uniq.append((t, u))
    return uniq

def pick_next_page_url(list_soup, current_url):
    """
    ✅ 핵심: 숫자 page++ 대신, 페이지네이션의 '다음' 링크를 실제로 따라갑니다.
    (180,181이 비는 현상 우회)
    """
    # 1) '다음', '›', '»' 같은 텍스트 우선
    for a in list_soup.select("a"):
        txt = (a.get_text(" ", strip=True) or "").replace(" ", "")
        href = (a.get("href") or "").strip()
        if not href:
            continue
        if txt in ["다음", "›", "»", ">", "NEXT", "Next"]:
            return urljoin(current_url, href)

    # 2) rel=next 우선
    a = list_soup.select_one('a[rel="next"]')
    if a and a.get("href"):
        return urljoin(current_url, a["href"])

    # 3) 페이지네이션에서 "현재 페이지" 다음 숫자 링크
    # (active/on/current 클래스는 스킨마다 다름)
    active = list_soup.select_one(".pagination .active, .paginate .active, .pg_current, .on, .active")
    if active:
        nxt = active.find_next("a")
        if nxt and nxt.get("href"):
            return urljoin(current_url, nxt["href"])

    return None

def crawl_from_179_to_0101():
    session = requests.Session()
    results = []
    seen_articles = set()
    seen_pages = set()

    url = START_URL
    while True:
        if url in seen_pages:
            print("⚠️ 같은 페이지를 반복하려 해서 종료:", url)
            break
        seen_pages.add(url)

        soup = get_soup(session, url)
        items = extract_article_items(soup, url)

        kept = 0
        stop_all = False

        for title, article_url in items:
            if article_url in seen_articles:
                continue
            seen_articles.add(article_url)

            d_obj, pack = parse_detail(get_soup(session, article_url))
            if not d_obj or not pack:
                continue

            d_str, view = pack

            # ✅ 2026-01-01 이전이면 종료
            if d_obj < TARGET_MIN_DATE:
                stop_all = True
                break

            # ✅ 상한(1/3)~하한(1/1)만 수집
            if d_obj > TARGET_MAX_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,
                "date": d_str,
                "view": view,
                "url": article_url,
            })
            kept += 1
            time.sleep(SLEEP_SEC)

        print(f"[page_url] kept={kept} items={len(items)} url={url}")

        if stop_all:
            print("✅ 2026-01-01 이전 글을 만나 종료합니다.")
            break

        next_url = pick_next_page_url(soup, url)
        if not next_url or next_url == url:
            print("⚠️ 다음 페이지 링크를 못 찾아 종료:", url)
            break

        url = next_url
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site","title","raw_date","date","view","url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=True)  # 1/1→1/3 순
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_from_179_to_0101()
    print("rows:", len(df))
    if len(df) > 0:
        print("min_date:", df["date"].min(), "max_date:", df["date"].max())

    df.to_csv("sheetline_qa_from_page179_to_20260101.csv", index=False, encoding="utf-8-sig")
    df.to_excel("sheetline_qa_from_page179_to_20260101.xlsx", index=False)
    print("saved: sheetline_qa_from_page179_to_20260101.csv / .xlsx")


[page_url] kept=15 items=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179
⚠️ 다음 페이지 링크를 못 찾아 종료: https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179
rows: 15
min_date: 2026-01-03 max_date: 2026-01-03
saved: sheetline_qa_from_page179_to_20260101.csv / .xlsx


In [19]:
import re
import time
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

START_URL = "https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179"
SITE_NAME = "시트라인"

TARGET_MIN_DATE = date(2026, 1, 1)   # ✅ 여기까지 내려가면 종료
TARGET_MAX_DATE = date(2026, 1, 3)   # ✅ 179페이지가 1/3이라 상한(원치 않으면 date.today()로 바꾸세요)

SLEEP_SEC = 0.35

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

DATE_RE = re.compile(r"(\d{4})[.\-\/](\d{1,2})[.\-\/](\d{1,2})")
VIEW_RE = re.compile(r"조회\s*([0-9,]+)")
ARTICLE_RE = re.compile(r"/article/.+/6/\d+")

def get_soup(session, url):
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def parse_detail(detail_soup):
    text = detail_soup.get_text(" ", strip=True)
    m = DATE_RE.search(text)
    if not m:
        return None, None
    y, mo, d = map(int, m.groups())
    d_obj = date(y, mo, d)
    d_str = f"{y:04d}-{mo:02d}-{d:02d}"

    m2 = VIEW_RE.search(text)
    view = int(m2.group(1).replace(",", "")) if m2 else None
    return d_obj, (d_str, view)

def extract_article_items(list_soup, base_url):
    anchors = list_soup.select('a[href*="/article/"][href*="/6/"]')
    items = []
    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href or not ARTICLE_RE.search(href):
            continue
        full = urljoin(base_url, href)
        title = a.get_text(" ", strip=True)
        if title:
            items.append((title, full))

    # 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for t, u in items:
        if u not in seen:
            seen.add(u)
            uniq.append((t, u))
    return uniq

def pick_next_page_url(list_soup, current_url):
    """
    ✅ 핵심: 숫자 page++ 대신, 페이지네이션의 '다음' 링크를 실제로 따라갑니다.
    (180,181이 비는 현상 우회)
    """
    # 1) '다음', '›', '»' 같은 텍스트 우선
    for a in list_soup.select("a"):
        txt = (a.get_text(" ", strip=True) or "").replace(" ", "")
        href = (a.get("href") or "").strip()
        if not href:
            continue
        if txt in ["다음", "›", "»", ">", "NEXT", "Next"]:
            return urljoin(current_url, href)

    # 2) rel=next 우선
    a = list_soup.select_one('a[rel="next"]')
    if a and a.get("href"):
        return urljoin(current_url, a["href"])

    # 3) 페이지네이션에서 "현재 페이지" 다음 숫자 링크
    # (active/on/current 클래스는 스킨마다 다름)
    active = list_soup.select_one(".pagination .active, .paginate .active, .pg_current, .on, .active")
    if active:
        nxt = active.find_next("a")
        if nxt and nxt.get("href"):
            return urljoin(current_url, nxt["href"])

    return None

def crawl_from_179_to_0101():
    session = requests.Session()
    results = []
    seen_articles = set()
    seen_pages = set()

    url = START_URL
    while True:
        if url in seen_pages:
            print("⚠️ 같은 페이지를 반복하려 해서 종료:", url)
            break
        seen_pages.add(url)

        soup = get_soup(session, url)
        items = extract_article_items(soup, url)

        kept = 0
        stop_all = False

        for title, article_url in items:
            if article_url in seen_articles:
                continue
            seen_articles.add(article_url)

            d_obj, pack = parse_detail(get_soup(session, article_url))
            if not d_obj or not pack:
                continue

            d_str, view = pack

            # ✅ 2026-01-01 이전이면 종료
            if d_obj < TARGET_MIN_DATE:
                stop_all = True
                break

            # ✅ 상한(1/3)~하한(1/1)만 수집
            if d_obj > TARGET_MAX_DATE:
                continue

            results.append({
                "site": SITE_NAME,
                "title": title,
                "raw_date": d_str,
                "date": d_str,
                "view": view,
                "url": article_url,
            })
            kept += 1
            time.sleep(SLEEP_SEC)

        print(f"[page_url] kept={kept} items={len(items)} url={url}")

        if stop_all:
            print("✅ 2026-01-01 이전 글을 만나 종료합니다.")
            break

        next_url = pick_next_page_url(soup, url)
        if not next_url or next_url == url:
            print("⚠️ 다음 페이지 링크를 못 찾아 종료:", url)
            break

        url = next_url
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(results, columns=["site","title","raw_date","date","view","url"])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date", ascending=True)  # 1/1→1/3 순
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df

if __name__ == "__main__":
    df = crawl_from_179_to_0101()
    print("rows:", len(df))
    if len(df) > 0:
        print("min_date:", df["date"].min(), "max_date:", df["date"].max())

    df.to_csv("sheetline_qa_from_page179_to_20260101.csv", index=False, encoding="utf-8-sig")
    df.to_excel("sheetline_qa_from_page179_to_20260101.xlsx", index=False)
    print("saved: sheetline_qa_from_page179_to_20260101.csv / .xlsx")


[page_url] kept=15 items=15 url=https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179
⚠️ 다음 페이지 링크를 못 찾아 종료: https://xn--oi2bs9sguddvo.com/board/%EC%83%81%ED%92%88-qa/6/?board_no=6&page=179
rows: 15
min_date: 2026-01-03 max_date: 2026-01-03
saved: sheetline_qa_from_page179_to_20260101.csv / .xlsx
